# Sentiment Analysis — EDA & Text Preprocessing

**Goal:** Understand the IMDb dataset visually, then clean the text so models can learn from it.

**Pipeline:**
```
Raw Data → EDA (explore) → Clean Text → Save Artifacts → Used by app.py
```

**What you'll learn:**
- How raw NLP data looks before cleaning
- Why each cleaning step matters
- How word frequency differs between positive and negative reviews

## 0. Install Dependencies

Run this cell once. After install, you can skip it on future runs.

In [ ]:
# !pip install datasets pandas matplotlib seaborn wordcloud scikit-learn tensorflow transformers

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re, string, os, pickle

# Set plot style for cleaner visuals
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('Imports OK')

## 2. Load Dataset

Using HuggingFace `datasets` to load IMDb — 25k train, 25k test reviews, binary labels (0=negative, 1=positive).

In [ ]:
from datasets import load_dataset

# Downloads ~80MB on first run, cached afterwards
raw = load_dataset('imdb')

# Convert to pandas for easier EDA
df_train = pd.DataFrame(raw['train'])
df_test  = pd.DataFrame(raw['test'])

print(f'Train: {len(df_train):,} rows')
print(f'Test : {len(df_test):,} rows')
df_train.head(3)

## 3. EDA — Exploratory Data Analysis

Before writing a single cleaning function, we need to **understand what we're working with**.

Questions to answer:
- Is the dataset balanced?
- How long are the reviews?
- What words appear most in positive vs negative reviews?
- Is there HTML / noise in the raw text?

### 3.1 Class Balance

Imbalanced classes → model might just predict the majority class and still get high accuracy. Always check this first.

In [ ]:
label_map = {0: 'Negative', 1: 'Positive'}
df_train['sentiment'] = df_train['label'].map(label_map)

counts = df_train['sentiment'].value_counts()
print(counts)
print(f'\nBalance ratio: {counts.min()/counts.max():.2f} (1.0 = perfectly balanced)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts.plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71'], edgecolor='black', rot=0)
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Number of Reviews')
for bar, count in zip(axes[0].patches, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{count:,}', ha='center', fontsize=11)

# Pie chart
axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#e74c3c','#2ecc71'], startangle=90)
axes[1].set_title('Class Distribution (%)')

plt.suptitle('IMDb Dataset — Class Balance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

# OBSERVATION: Perfect 50/50 balance — no class imbalance to worry about

### 3.2 Review Length Distribution

Why this matters:
- Very short reviews → less signal
- Very long reviews → need truncation for models
- Length difference between positive/negative → could be a feature itself

In [ ]:
# Character-level and word-level lengths
df_train['char_len'] = df_train['text'].str.len()
df_train['word_len'] = df_train['text'].str.split().str.len()

print('=== Character Length Stats ===')
print(df_train.groupby('sentiment')['char_len'].describe().round(1))
print('\n=== Word Count Stats ===')
print(df_train.groupby('sentiment')['word_len'].describe().round(1))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, (col, label) in enumerate([('char_len','Characters'), ('word_len','Words')]):
    # Histogram by class
    for sentiment, color in [('Positive','#2ecc71'), ('Negative','#e74c3c')]:
        subset = df_train[df_train['sentiment'] == sentiment][col]
        axes[i][0].hist(subset.clip(upper=subset.quantile(0.99)),
                        bins=60, alpha=0.6, color=color, label=sentiment)
    axes[i][0].set_title(f'Distribution of {label} per Review')
    axes[i][0].set_xlabel(f'Number of {label}')
    axes[i][0].set_ylabel('Count')
    axes[i][0].legend()

    # Box plot
    df_train.boxplot(column=col, by='sentiment', ax=axes[i][1],
                     boxprops=dict(color='steelblue'),
                     showfliers=False)  # hide outliers for readability
    axes[i][1].set_title(f'Box Plot — {label} by Sentiment')
    axes[i][1].set_xlabel('Sentiment')

plt.suptitle('Review Length Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_review_length.png', dpi=150, bbox_inches='tight')
plt.show()

# For model input: use max_len around 256-512 words (covers ~95% of reviews)

### 3.3 Sample Raw Reviews — Spot the Noise

Looking at raw text shows us exactly what we need to clean.

In [ ]:
print('=== RAW TEXT SAMPLES ===')
for i, row in df_train.sample(3, random_state=42).iterrows():
    print(f'\n[{row["sentiment"]}] (first 300 chars):')
    print('-' * 60)
    print(row['text'][:300])

# You'll notice:
# - HTML tags like <br />
# - Punctuation scattered everywhere
# - Numbers that carry no meaning
# - Stopwords (the, is, a) that add noise

### 3.4 Top Words — Before Cleaning

Seeing what dominates before cleaning motivates WHY we remove stopwords and punctuation.

In [ ]:
from collections import Counter

def top_words(texts, n=20):
    """Count most frequent words across a list of texts."""
    all_words = ' '.join(texts).lower().split()
    return Counter(all_words).most_common(n)

pos_texts = df_train[df_train['label'] == 1]['text'].tolist()
neg_texts = df_train[df_train['label'] == 0]['text'].tolist()

pos_top = top_words(pos_texts)
neg_top = top_words(neg_texts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, top, title, color in [
    (axes[0], pos_top, 'Top 20 Words — POSITIVE', '#2ecc71'),
    (axes[1], neg_top, 'Top 20 Words — NEGATIVE', '#e74c3c')
]:
    words, counts = zip(*top)
    ax.barh(words[::-1], counts[::-1], color=color, edgecolor='black', alpha=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.suptitle('Most Frequent Words BEFORE Cleaning', fontsize=13)
plt.tight_layout()
plt.savefig('eda_top_words_before.png', dpi=150, bbox_inches='tight')
plt.show()

# OBSERVATION: Top words are almost identical (the, a, and, is...)
# These stopwords contain ZERO sentiment signal — must remove them

## 4. Text Cleaning

Each step below removes a specific type of noise. We'll see the before/after for each.

In [ ]:
# Import our shared cleaning functions from utils.py
import sys
sys.path.insert(0, '.')  # add current folder to path
from utils import clean_text, remove_html, remove_punctuation, remove_stopwords

print('utils.py loaded OK')

### 4.1 Step-by-Step Walkthrough on a Single Review

Seeing each step on one example makes the pipeline concrete.

In [ ]:
import re, string

sample = df_train.iloc[0]['text']
print('--- ORIGINAL ---')
print(sample[:400])

step1 = sample.lower()
print('\n--- Step 1: Lowercase ---')
print(step1[:200])

step2 = re.sub(r'<[^>]+>', ' ', step1)   # remove HTML
print('\n--- Step 2: Remove HTML tags ---')
print(step2[:200])

step3 = step2.translate(str.maketrans('', '', string.punctuation))
print('\n--- Step 3: Remove punctuation ---')
print(step3[:200])

step4 = re.sub(r'\b\d+\b', '', step3)   # remove standalone numbers
print('\n--- Step 4: Remove numbers ---')
print(step4[:200])

# Stopword removal
step5 = remove_stopwords(step4)
print('\n--- Step 5: Remove stopwords ---')
print(step5[:200])

step6 = re.sub(r'\s+', ' ', step5).strip()
print('\n--- Step 6: Collapse whitespace ---')
print(step6[:200])

print(f'\nOriginal word count : {len(sample.split())}')
print(f'Cleaned word count  : {len(step6.split())}')

### 4.2 Apply Cleaning to Full Dataset

In [ ]:
print('Cleaning train set...')
df_train['clean_text'] = df_train['text'].apply(clean_text)

df_test['sentiment'] = df_test['label'].map(label_map)
print('Cleaning test set...')
df_test['clean_text'] = df_test['text'].apply(clean_text)

print('Done!')
df_train[['text','clean_text','label']].head(3)

### 4.3 Top Words — After Cleaning

Now we should see meaningful sentiment words dominate each class.

In [ ]:
pos_clean = df_train[df_train['label'] == 1]['clean_text'].tolist()
neg_clean = df_train[df_train['label'] == 0]['clean_text'].tolist()

pos_top_clean = top_words(pos_clean)
neg_top_clean = top_words(neg_clean)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, top, title, color in [
    (axes[0], pos_top_clean, 'Top 20 Words — POSITIVE (cleaned)', '#2ecc71'),
    (axes[1], neg_top_clean, 'Top 20 Words — NEGATIVE (cleaned)', '#e74c3c')
]:
    words, counts = zip(*top)
    ax.barh(words[::-1], counts[::-1], color=color, edgecolor='black', alpha=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.suptitle('Most Frequent Words AFTER Cleaning', fontsize=13)
plt.tight_layout()
plt.savefig('eda_top_words_after.png', dpi=150, bbox_inches='tight')
plt.show()

# Now you can see actual sentiment words:
# Positive: great, best, love, excellent...
# Negative: bad, worst, waste, awful...

### 4.4 WordCloud Visualization

WordCloud = visual frequency map. Bigger word = appears more often.

In [ ]:
try:
    from wordcloud import WordCloud

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for ax, texts, title, cmap in [
        (axes[0], pos_clean, 'Positive Reviews', 'Greens'),
        (axes[1], neg_clean, 'Negative Reviews', 'Reds')
    ]:
        wc = WordCloud(
            width=800, height=400,
            background_color='white',
            colormap=cmap,
            max_words=100
        ).generate(' '.join(texts))

        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(title, fontsize=14, fontweight='bold')

    plt.suptitle('WordCloud — After Cleaning', fontsize=15)
    plt.tight_layout()
    plt.savefig('eda_wordcloud.png', dpi=150, bbox_inches='tight')
    plt.show()

except ImportError:
    print('wordcloud not installed. Run: pip install wordcloud')

### 4.5 Review Length After Cleaning

Cleaning reduces length — verify the distribution is still usable.

In [ ]:
df_train['clean_word_len'] = df_train['clean_text'].str.split().str.len()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_train['word_len'].clip(upper=800),   bins=60, alpha=0.6, label='Before cleaning', color='steelblue')
ax.hist(df_train['clean_word_len'].clip(upper=800), bins=60, alpha=0.6, label='After cleaning', color='orange')
ax.axvline(256, color='red', linestyle='--', label='max_len=256 cutoff')
ax.set_title('Word Count Distribution — Before vs After Cleaning')
ax.set_xlabel('Word Count')
ax.set_ylabel('Number of Reviews')
ax.legend()
plt.tight_layout()
plt.savefig('eda_length_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

coverage = (df_train['clean_word_len'] <= 256).mean()
print(f'Reviews fully covered by max_len=256: {coverage:.1%}')
# If coverage < 90%, consider increasing max_len

## 5. Prepare & Save Artifacts

Save the cleaned data and fitted preprocessing objects so `app.py` can load them without re-running this notebook.

In [ ]:
# Save cleaned DataFrames
df_train[['text','clean_text','label','sentiment']].to_csv('cleaned_train.csv', index=False)
df_test[['text','clean_text','label','sentiment']].to_csv('cleaned_test.csv', index=False)
print('Saved: cleaned_train.csv, cleaned_test.csv')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 20_000
MAX_LEN    = 256

# --- TF-IDF vectorizer (for MLP Stage 1) ---
tfidf = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2))
tfidf.fit(df_train['clean_text'])
print(f'TF-IDF vocab size: {len(tfidf.vocabulary_):,}')

# --- Keras tokenizer (for LSTM Stage 2) ---
keras_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
keras_tokenizer.fit_on_texts(df_train['clean_text'])
print(f'Keras tokenizer vocab: {len(keras_tokenizer.word_index):,} unique tokens')

# --- Save both ---
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

import json
tokenizer_config = keras_tokenizer.to_json()
with open('keras_tokenizer.json', 'w') as f:
    f.write(tokenizer_config)

# Save constants
with open('preprocess_config.json', 'w') as f:
    json.dump({'VOCAB_SIZE': VOCAB_SIZE, 'MAX_LEN': MAX_LEN,
               'TFIDF_FEATURES': 10_000}, f)

print('\nSaved artifacts:')
print('  tfidf_vectorizer.pkl')
print('  keras_tokenizer.json')
print('  preprocess_config.json')

## 6. EDA Summary

| Finding | Implication |
|---------|------------|
| 50/50 class balance | No need for oversampling / class weights |
| Mean review length ~230 words (clean) | max_len=256 covers ~90%+ of reviews |
| Before cleaning: top words = stopwords | Stopword removal is critical for MLP/TFIDF |
| After cleaning: words differ by class | Signal is there for models to learn from |
| HTML tags present | Must strip before tokenization |

**Artifacts saved** → ready for `app.py` to load and use for model training + inference.